# Find Incomplete Sessions

Continuation pass for the build waves job. Runs as `w<wave>_find_incomplete[k]` after a wave's sessions (and after each earlier continuation pass), with `run_if` ALL_DONE.

A Genie Code session that stops at the step limit still ends its `for_each` iteration SUCCESS, and a repair cannot re-run a SUCCESS iteration. This notebook lists the wave's sessions whose status file (`<build_root>/<status_dir>/<run_id>/<session_id>.json`) is **missing**, **unreadable** (bad JSON or not an object) or has no final status (anything but `pass`, `fail`, `blocked`, `error`; reported as **in_progress**), and sets two task values:

* `incomplete`: a JSON list of `{session_id, title}`, the inputs of `w<wave>_continue[k]` (runner with `continuation=k`, which resumes from `progress.next`)
* `has_incomplete`: `"true"` or `"false"`, read by the `w<wave>_has_incomplete[k]` If/else task. When false, the continue task is Excluded and the integrator still runs.

The rule is `session_outcome` in `session_status.py`, the same one the gate uses. Final statuses are never re-run.

With branch sync on (`git` / `hybrid` workflows), nothing is re-run when the wave's `w<wave>_sync` task in this run ended in a state other than `SUCCESS` (it crashed, so the folder may not have the branch head).

In [0]:
dbutils.widgets.text("wave", "", "Wave id")
dbutils.widgets.text("continuation", "1", "Continuation pass (1-based)")
dbutils.widgets.text("run_id", "", "Job run id")
dbutils.widgets.text("build_root", "", "Build root (project Git folder)")
dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")

wave = dbutils.widgets.get("wave").strip()
continuation = dbutils.widgets.get("continuation").strip() or "1"
run_id = dbutils.widgets.get("run_id").strip()
build_root = dbutils.widgets.get("build_root").strip()
manifest_path = dbutils.widgets.get("manifest_path").strip()

if not wave or not run_id or not build_root or not manifest_path:
    raise ValueError(f"wave, run_id, build_root and manifest_path are required "
                     f"(got wave={wave!r}, run_id={run_id!r}, build_root={build_root!r}, manifest_path={manifest_path!r})")

print(f"wave={wave} continuation={continuation} run_id={run_id}")

In [0]:
import json
import os
import sys

# waves_manifest.py and session_status.py sit next to this notebook (cwd for workspace notebooks).
sys.path.insert(0, os.getcwd())
from waves_manifest import is_archived, sync_enabled, sync_keys
from session_status import FINAL_STATUSES, incomplete_sessions

with open(manifest_path) as f:
    manifest = json.load(f)

wave_cfg = next((w for w in manifest.get("waves") or [] if str(w.get("wave_id")) == wave), None)
if wave_cfg is None:
    raise ValueError(f"Wave {wave!r} not found in {manifest_path}")
if is_archived(wave_cfg):
    raise ValueError(f"Wave {wave!r} is archived; re-run setup so the build job has no tasks for it")

status_dir = os.path.join(build_root, manifest["status_dir"], run_id)
incomplete = incomplete_sessions(wave_cfg, status_dir)

# Branch sync guard: w<wave>_sync never raises, but if it crashed (FAILED / TIMEDOUT) the wave's
# sessions are Upstream failed and this run_if ALL_DONE task still runs. Do not start sessions on
# a folder that was not synced; build_verdict fails the run on the sync task's state.
if incomplete and sync_enabled(manifest):
    sync_key = sync_keys(f"w{wave}")["sync"]
    try:
        from databricks.sdk import WorkspaceClient
        from gate_helpers import latest_task_states
        from jobs_api import get_run
        sync_state = latest_task_states(get_run(WorkspaceClient().api_client, run_id).get("tasks")).get(sync_key)
    except Exception as e:  # interactive run with a fake run_id, or an API error
        print(f"WARN: could not read {sync_key}'s state ({type(e).__name__}: {str(e)[:200]}); not checking the sync")
        sync_state = None
    if sync_state not in (None, "SUCCESS"):
        print(f"{sync_key} ended {sync_state}: the Git folder was not synced, so no continuation pass "
              f"({len(incomplete)} incomplete session(s) left as they are)")
        incomplete = []

print(f"status dir: {status_dir}")
print(f"sessions in wave: {len(wave_cfg.get('sessions') or [])}, incomplete: {len(incomplete)} "
      f"(final statuses: {sorted(FINAL_STATUSES)})")
for i in incomplete:
    print(f"  {i['session_id']}: {i['reason']} ({i['title']})")

# for_each inputs for w<wave>_continue[k]: same shape as the w<wave>_sessions inputs.
inputs = [{"session_id": i["session_id"], "title": i["title"]} for i in incomplete]
# No try/except: if these fail in a job, w<wave>_has_incomplete cannot resolve its value and the
# continuation would be skipped silently. Outside a job, taskValues.set does nothing.
dbutils.jobs.taskValues.set(key="incomplete", value=inputs)
dbutils.jobs.taskValues.set(key="has_incomplete", value="true" if inputs else "false")

dbutils.notebook.exit(json.dumps({"wave": wave, "continuation": continuation, "incomplete": incomplete}))